### Data Understanding

**Goal**:  
Understand the structure, quality, and key patterns in the dataset before building product metrics (LTV, CAC, churn, etc.)

In [ ]:
# Set working directory
import local_config
from local_config import directory_path
import os
os.chdir(directory_path)

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# display settings
pd.set_option('display.max_columns', None)

In [ ]:
# Load data
accounts = pd.read_csv('data/accounts.csv')
subscriptions = pd.read_csv('data/subscriptions.csv')
feature_usage = pd.read_csv('data/feature_usage.csv')
support_tickets = pd.read_csv('data/support_tickets.csv')
churn_events = pd.read_csv('data/churn_events.csv')


### Initial Overview

#### Shape & Preview
Check dataset sizes and preview records.

In [ ]:
accounts.shape
accounts.head()

Accounts dataset: (500, 10)

In [ ]:
subscriptions.shape
subscriptions.head()

Subscriptions dataset: (5000, 14)

The number of subscription records significantly exceeds the number of accounts,
which suggests multiple subscription lifecycle events per account 
(e.g., renewals, upgrades, downgrades).

In [ ]:
feature_usage.shape
feature_usage.head()

Feature usage dataset: (25000, 8)

In [ ]:
support_tickets.shape
support_tickets.head()

Support tickets dataset: (2000, 9)

In [ ]:
churn_events.shape
churn_events.head()

Churn events dataset: (600, 9)

### Columns & Data Types
Inspect column types and identify fields that require transformation.

In [ ]:
accounts.info()

`signup_date` is stored as an object and should be converted to datetime.

In [ ]:
subscriptions.info()

`start_date` and `end_date` should be converted to datetime.

In [ ]:
feature_usage.info()

`usage_date` should be converted to datetime.

In [ ]:
support_tickets.info()

`submitted_at` and `closed_at` should be converted to datetime.

In [ ]:
churn_events.info()

`churn_date` should be converted to datetime.

### Data Quality Checks

In [ ]:
accounts.isnull().sum()
subscriptions.isnull().sum()
feature_usage.isnull().sum()
support_tickets.isnull().sum()
churn_events.isnull().sum()

- No missing values in most tables  
- Missing `end_date` likely indicates active subscriptions  
- Missing `satisfaction_score` may limit support analysis  
- Missing `feedback_text` is not critical  

In [ ]:
accounts.duplicated().sum()
subscriptions.duplicated().sum()
feature_usage.duplicated().sum()
support_tickets.duplicated().sum()
churn_events.duplicated().sum()

No duplicates found in any table.

In [ ]:
accounts.describe()
subscriptions.describe()

Data appears consistent:
- No negative values in revenue or seats
- No obvious anomalies

In [ ]:
accounts['signup_date'].min(), accounts['signup_date'].max()
subscriptions['start_date'].min(), subscriptions['start_date'].max()
churn_events['churn_date'].min(), churn_events['churn_date'].max()

The dataset covers a 2-year period (2023–2024).

### Accounts Exploration

In [ ]:
accounts['country'].value_counts()
accounts['country'].value_counts(normalize=True) * 100

Most users are from the US (~58%).

In [ ]:
accounts['plan_tier'].value_counts()

Plan distribution is relatively balanced.

In [ ]:
accounts['referral_source'].value_counts(normalize=True) * 100

Referral sources are evenly distributed → good basis for channel comparison.

In [ ]:
accounts['is_trial'].value_counts(normalize=True) * 100

~19% of users are on trial → enables funnel analysis.

### Revenue Exploration (subscriptions)

In [ ]:
# MRR distribution
sns.histplot(subscriptions['mrr_amount'], bins=50)
plt.title('MRR Distribution')
plt.show()

In [ ]:
# ARR distribution
sns.histplot(subscriptions['arr_amount'], bins=50)
plt.title('ARR Distribution')
plt.show()

Revenue is right-skewed:
- Many low-paying users
- Few high-value users (long tail)

In [ ]:
subscriptions['mrr_amount'].mean()

Average MRR provides a rough estimate of ARPU.

In [ ]:
subscriptions['billing_frequency'].value_counts(normalize=True) * 100

Monthly and annual billing are evenly distributed.

### Churn Exploration

In [ ]:
subscriptions['churn_flag'].mean()
subscriptions.groupby('plan_tier')['churn_flag'].mean()

Churn rate is ~10% and consistent across plans.

In [ ]:
churn_events['reason_code'].value_counts()

Most churn is related to product features rather than pricing.

### Feature usage

In [ ]:
feature_usage['feature_name'].value_counts().head()

 Feature usage data is available and can be used later to analyze engagement vs churn.

### Data Relationships

In [ ]:
df = subscriptions.merge(accounts, on='account_id', how='left')
df['account_id'].isnull().sum()

All subscriptions are successfully linked to accounts.

### Key Observations & Insights

- Multiple subscription records per account
- Revenue is concentrated in a small number of users
- Churn is moderate and not pricing-driven
- Channels are evenly distributed → good for CAC analysis
- Dataset supports full unit economics modeling

### Next Steps

- Data cleaning
- Marketing channel mapping
- Synthetic CAC generation
- SQL modeling
- Metrics calculation